# PCM Training — Benchmark-Grounded Dataset

This notebook trains the Policy Compliance Model (PCM) on a dataset built directly from the local ST-WebAgentBench task and policy catalogue.

Pipeline:
1. Install dependencies
2. Build `data/benchmark_grounded/{train,val,test,challenge}.jsonl`
3. Fine-tune DeBERTa
4. Evaluate on held-out test and challenge splits
5. Run sanity probes before exporting the checkpoint


In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.44.0', 'torch', 'scikit-learn', 'scipy', 'sentencepiece', 'tqdm'
])
print('Dependencies installed.')


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

ROOT = Path('.').resolve()
DATA_DIR = ROOT / 'data' / 'benchmark_grounded'
MODEL_DIR = ROOT / 'models' / 'pcm_benchmark_grounded'
RESULTS_PATH = ROOT / 'results' / 'benchmark_grounded_test_metrics.json'
CATALOG = ROOT / 'ST-WebAgentBench' / 'stwebagentbench' / 'test.raw.json'

print('ROOT       :', ROOT)
print('CATALOG    :', CATALOG)
print('DATA_DIR   :', DATA_DIR)
print('MODEL_DIR  :', MODEL_DIR)
print('RESULTS    :', RESULTS_PATH)


In [ ]:
subprocess.check_call([
    sys.executable,
    str(ROOT / 'build_benchmark_grounded_pcm_dataset.py'),
    '--catalog', str(CATALOG),
    '--output_dir', str(DATA_DIR),
    '--seed', '42',
])

manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
print(json.dumps(manifest['samples'], indent=2))
print(json.dumps(manifest['label_counts'], indent=2))


In [ ]:
import itertools

for split_name in ['train', 'val', 'test', 'challenge']:
    path = DATA_DIR / f'{split_name}.jsonl'
    rows = [json.loads(line) for line in itertools.islice(open(path), 3)]
    print('\n===', split_name.upper(), '===')
    for row in rows:
        print({k: row[k] for k in ['dimension', 'policy_template_id', 'policy', 'action', 'label']})


In [ ]:
subprocess.check_call([
    sys.executable,
    str(ROOT / 'train_pcm.py'),
    '--train', str(DATA_DIR / 'train.jsonl'),
    '--val', str(DATA_DIR / 'val.jsonl'),
    '--output_dir', str(MODEL_DIR),
    '--model_name', 'microsoft/deberta-v3-base',
    '--epochs', '5',
    '--batch_size', '16',
    '--max_len', '512',
    '--patience', '2',
])


In [ ]:
subprocess.check_call([
    sys.executable,
    str(ROOT / 'evaluate_pcm.py'),
    '--model', str(MODEL_DIR / 'best'),
    '--test', str(DATA_DIR / 'test.jsonl'),
    '--challenge', str(DATA_DIR / 'challenge.jsonl'),
    '--output', str(RESULTS_PATH),
    '--threshold', '0.5',
])

results = json.loads(RESULTS_PATH.read_text())
print(json.dumps(results.get('overall', {}), indent=2))
print(json.dumps(results.get('challenge_overall', {}), indent=2))


In [ ]:
import itertools
import sys

sys.path.insert(0, str(ROOT))
from policy_compliant_agent import PCMClassifier

pcm = PCMClassifier(str(MODEL_DIR / 'best'), device='cpu')
for row in [json.loads(line) for line in itertools.islice(open(DATA_DIR / 'sanity_probes.jsonl'), 6)]:
    prob = pcm.predict(row['policy'], row['context'], row['action'])
    print({
        'label': row['label'],
        'prob': round(prob, 4),
        'policy': row['policy'],
        'action': row['action'],
    })


In [ ]:
archive_base = ROOT / 'results' / 'pcm_benchmark_grounded_artifact'
if archive_base.with_suffix('.zip').exists():
    archive_base.with_suffix('.zip').unlink()
shutil.make_archive(str(archive_base), 'zip', root_dir=str(ROOT), base_dir='models/pcm_benchmark_grounded')
print('Checkpoint archive:', archive_base.with_suffix('.zip'))
print('Metrics file      :', RESULTS_PATH)
